In [0]:
%pip install -U --quiet databricks-langchain==0.6.0 mlflow[databricks]==3.4.0  langchain==0.3.27 langchain_core==0.3.74 bs4 langchain_community markdownify docling pypdf2 pypdf

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
%run "../_config/Config_Unity_Catalog"

## PDF Sources

In [0]:
# PDF volume path
path_volume = f"/Volumes/{catalog}/{schema}/raw_data/pdf/"
pdf_list = dbutils.fs.ls(path_volume)
print("PDF list : ") 
for pdf_infos in pdf_list[:2] : 
    print(pdf_infos)

PDF list : 
FileInfo(path='dbfs:/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf', name='product_catalog.pdf', size=13874, modificationTime=1788182263000)
FileInfo(path='dbfs:/Volumes/demo/demo/raw_data/pdf/return_policy.pdf', name='return_policy.pdf', size=12480, modificationTime=1788182263000)


## Chunking with recursive splitter

In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from pypdf.errors import PdfStreamError
import time

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

def read_file_header(file_path, num_bytes=32):
    with open(file_path, "rb") as f:
        return f.read(num_bytes)

def classic_pdf_splitter(pdf_path) : 
    """
    Load and split a PDF using PyPDFLoader and RecursiveCharacterTextSplitter.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        List of Document chunks with metadata
    """
    print(f"Processing {pdf_path}")
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    #Return the chunked pages
    return text_splitter.split_documents(pages)

def create_classic_documents(path_volume) : 
    """
    For each PDF, add the Document results from classic_pdf_splitter
    to the list of documents extracted from the sources. 
    
    Args:
        path_volume: Path to the volume containing PDF files
    
    Returns:
        List of all document chunks from all PDFs
    """
    documents = []
    start_time = time.time()
    for pdf_infos in dbutils.fs.ls(path_volume) :
        pdf_path = f"{path_volume}{pdf_infos.name}"
        print(f"Processing {pdf_infos.name}")
        header = read_file_header(pdf_path)
        if not header.startswith(b"%PDF-"):
            print(f"Skipping {pdf_infos.name} (not a PDF file, header={header!r})")
            continue
        try:
            documents += classic_pdf_splitter(pdf_path)
        except PdfStreamError as e:
            print(f"Skipping {pdf_infos.name} (corrupted PDF): {e}")
    
    end_time = time.time() - start_time

    # Evaluation of the processing time
    print(f"classic_pdf_splitter took {end_time} seconds to process classic documents")
    return documents


documents = create_classic_documents(path_volume)
print(f"{len(documents)} chunks ! ")

Processing product_catalog.pdf
Processing /Volumes/demo/demo/raw_data/pdf/product_catalog.pdf
Processing return_policy.pdf
Processing /Volumes/demo/demo/raw_data/pdf/return_policy.pdf
Processing shipping_guide.pdf
Processing /Volumes/demo/demo/raw_data/pdf/shipping_guide.pdf
Processing technical_faq.pdf
Processing /Volumes/demo/demo/raw_data/pdf/technical_faq.pdf
classic_pdf_splitter took 1.1575837135314941 seconds to process classic documents
63 chunks ! 


In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pathlib import Path
schema = StructType([
    StructField("content", StringType(), True),
    StructField("source", StringType(), True),
    StructField("source_name", StringType(), True),
])
doc_dict = [{'content' : doc.page_content,
              'source': doc.metadata['source'],
              'source_name' : Path(doc.metadata['source']).name,
              } for doc in documents]

df_spark = spark.createDataFrame(doc_dict, schema=schema)
display(df_spark.limit(2))

content,source,source_name
"Product Catalog 2024 Premium Electronics & Accessories Monitors & Displays UltraView 4K Monitor 27"" - Model UV-27-4K Price: $399.99 | SKU: MON-UV27-001 Technical Specifications:  Display: 27-inch IPS panel, 3840 x 2160 resolution (4K UHD)  Refresh Rate: 60Hz  Response Time: 5ms (GtG)  Brightness: 350 cd/m²  Contrast Ratio: 1000:1  Color Gamut: 99% sRGB, 95% DCI-P3  HDR Support: HDR10  Connectivity: 2x HDMI 2.0, 1x DisplayPort 1.4, 4x USB 3.0  Stand: Height adjustable, tilt, swivel, pivot  VESA Mount: 100x100mm  Dimensions: 24.1"" x 18.5"" x 8.9""  Weight: 6.5 kg (14.3 lbs) Key Features:  Flicker-free technology reduces eye strain  Low blue light mode for extended viewing  Built-in KVM switch for multi-computer setup  Picture-in-Picture and Picture-by-Picture modes  On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf
" On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide Warranty: 3-year manufacturer warranty covering defects in materials and workmanship Compatible With: Windows 10/11, macOS 10.13+, Linux, PlayStation 5, Xbox Series X/S Best For: Content creators, photographers, graphic designers, office productivity, casual gaming Portable Monitor 15.6"" - Model PM-156-FHD Price: $299.99 | SKU: MON-PM156-002 Technical Specifications:  Display: 15.6-inch IPS, 1920 x 1080 (Full HD)  Brightness: 300 cd/m²",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf


In [0]:
# Define the name of your UC table (catalog.schema.table)
table_name = "pdf_document_raw"

# Create the UC table
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM pdf_document_raw limit 2;

content,source,source_name
"Product Catalog 2024 Premium Electronics & Accessories Monitors & Displays UltraView 4K Monitor 27"" - Model UV-27-4K Price: $399.99 | SKU: MON-UV27-001 Technical Specifications:  Display: 27-inch IPS panel, 3840 x 2160 resolution (4K UHD)  Refresh Rate: 60Hz  Response Time: 5ms (GtG)  Brightness: 350 cd/m²  Contrast Ratio: 1000:1  Color Gamut: 99% sRGB, 95% DCI-P3  HDR Support: HDR10  Connectivity: 2x HDMI 2.0, 1x DisplayPort 1.4, 4x USB 3.0  Stand: Height adjustable, tilt, swivel, pivot  VESA Mount: 100x100mm  Dimensions: 24.1"" x 18.5"" x 8.9""  Weight: 6.5 kg (14.3 lbs) Key Features:  Flicker-free technology reduces eye strain  Low blue light mode for extended viewing  Built-in KVM switch for multi-computer setup  Picture-in-Picture and Picture-by-Picture modes  On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf
" On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide Warranty: 3-year manufacturer warranty covering defects in materials and workmanship Compatible With: Windows 10/11, macOS 10.13+, Linux, PlayStation 5, Xbox Series X/S Best For: Content creators, photographers, graphic designers, office productivity, casual gaming Portable Monitor 15.6"" - Model PM-156-FHD Price: $299.99 | SKU: MON-PM156-002 Technical Specifications:  Display: 15.6-inch IPS, 1920 x 1080 (Full HD)  Brightness: 300 cd/m²",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf


## Chunking with docling

In [0]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker
import time

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False

doc_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)
chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=512,
    include_section_info=True,
)

def docling_pdf_splitter(pdf_path) : 
    """
    Load and split a PDF using Docling's HybridChunker.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        List of dictionaries containing chunk content and metadata
    """
    print(f"Processing {pdf_path}")

    doc_result = doc_converter.convert(pdf_path).document
    print(f"doc_result {doc_result.name}")
    chunk_iter = chunker.chunk(dl_doc=doc_result)
    data_docling = []
    for i, chunk in enumerate(chunk_iter):
        enriched_text = chunker.contextualize(chunk=chunk)
        row = {
            #'content': chunk,
            'content' : enriched_text,
            'source': pdf_path, 
            'source_name' : Path(pdf_path).name, 
        }
        data_docling.append(row)
    print(f"=> {i} Chunks ")
    return data_docling

def create_docling_documents(path_volume) : 
    """
    For each PDF, add the Document results from docling_pdf_splitter
    to the list of documents extracted from the sources. 
    
    Args:
        path_volume: Path to the volume containing PDF files
    
    Returns:
        List of all document chunks from all PDFs
    """
    documents = []
    start_time = time.time()
    for pdf_infos in dbutils.fs.ls(path_volume) :
        print(f"Processing {pdf_infos.name}")
        documents += docling_pdf_splitter(f"{path_volume}{pdf_infos.name}")
    
    end_time = time.time() - start_time

    # Evaluation of the processing time
    print(f"docling_pdf_splitter took {end_time} seconds to process docling documents")
    return documents


docling_documents = create_docling_documents(path_volume)
print(f"{len(docling_documents)} chunks ! ")
     

Processing product_catalog.pdf
Processing /Volumes/demo/demo/raw_data/pdf/product_catalog.pdf


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

doc_result product_catalog
=> 47 Chunks 
Processing return_policy.pdf
Processing /Volumes/demo/demo/raw_data/pdf/return_policy.pdf
doc_result return_policy
=> 41 Chunks 
Processing shipping_guide.pdf
Processing /Volumes/demo/demo/raw_data/pdf/shipping_guide.pdf
doc_result shipping_guide
=> 43 Chunks 
Processing technical_faq.pdf
Processing /Volumes/demo/demo/raw_data/pdf/technical_faq.pdf
doc_result technical_faq
=> 70 Chunks 
docling_pdf_splitter took 82.21959090232849 seconds to process docling documents
205 chunks ! 


In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("content", StringType(), True),
    StructField("source", StringType(), True),
    StructField("source_name", StringType(), True),
])
df_spark_docling = spark.createDataFrame(docling_documents, schema=schema)
display(df_spark_docling.limit(2))

content,source,source_name
Product Catalog 2024 Premium Electronics & Accessories,/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf
Price: $399.99 | SKU: MON-UV27-001,/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf


In [0]:
%sql
CREATE OR REPLACE TABLE pdf_document_docling (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  content STRING,
  source STRING,
  source_name STRING,
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.feature.allowColumnDefaults' = 'enabled'
);

In [0]:
# Define the name of your UC table (catalog.schema.table)
table_name = "pdf_document_docling"

# Create the UC table
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM pdf_document_docling limit 2;

content,source,source_name
"Product Catalog 2024 Premium Electronics & Accessories Monitors & Displays UltraView 4K Monitor 27"" - Model UV-27-4K Price: $399.99 | SKU: MON-UV27-001 Technical Specifications:  Display: 27-inch IPS panel, 3840 x 2160 resolution (4K UHD)  Refresh Rate: 60Hz  Response Time: 5ms (GtG)  Brightness: 350 cd/m²  Contrast Ratio: 1000:1  Color Gamut: 99% sRGB, 95% DCI-P3  HDR Support: HDR10  Connectivity: 2x HDMI 2.0, 1x DisplayPort 1.4, 4x USB 3.0  Stand: Height adjustable, tilt, swivel, pivot  VESA Mount: 100x100mm  Dimensions: 24.1"" x 18.5"" x 8.9""  Weight: 6.5 kg (14.3 lbs) Key Features:  Flicker-free technology reduces eye strain  Low blue light mode for extended viewing  Built-in KVM switch for multi-computer setup  Picture-in-Picture and Picture-by-Picture modes  On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf
" On-screen display (OSD) with 5-way joystick control In The Box: Monitor, power cable, HDMI cable, DisplayPort cable, USB upstream cable, quick start guide Warranty: 3-year manufacturer warranty covering defects in materials and workmanship Compatible With: Windows 10/11, macOS 10.13+, Linux, PlayStation 5, Xbox Series X/S Best For: Content creators, photographers, graphic designers, office productivity, casual gaming Portable Monitor 15.6"" - Model PM-156-FHD Price: $299.99 | SKU: MON-PM156-002 Technical Specifications:  Display: 15.6-inch IPS, 1920 x 1080 (Full HD)  Brightness: 300 cd/m²",/Volumes/demo/demo/raw_data/pdf/product_catalog.pdf,product_catalog.pdf
